### Imports

In [1]:
import torch
import torch.nn as nn
import copy
import random
import numpy as np
import torch.optim as optim
from tqdm.notebook import tqdm, trange
from torchvision import datasets, transforms
import time
import gc
#from GPUtil import showUtilization as gpu_usage
#from GPUtil import getGPUs
from torchvision.transforms import v2
import os, ctypes
from IPython.core.display import HTML
import os
import shutil
import json
from pathlib import Path
#import opendatasets as od
from torch.utils.data import Dataset, DataLoader
import re
from matplotlib import pyplot as plt

# For the correct functioning of tqdm
display(HTML("""
    <style>
        .jp-OutputArea-child:has(.jp-OutputArea-prompt:empty) {
              padding: 0 !important;
        }
    </style>
"""))

# To supress pesky warnings
import warnings
warnings.filterwarnings("ignore")

### Reproducibility

In [2]:
def set_seed(seed_value=42):
    np.random.seed(seed_value) # cpu vars
    random.seed(seed_value) # Python
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    os.environ['CUBLAS_WORKSPACE_CONFIG']=":4096:8"
    torch.manual_seed(seed_value) # cpu  vars
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.cuda.manual_seed_all(seed_value) # gpu vars
    torch.backends.cudnn.deterministic = True  #needed
    torch.backends.cudnn.benchmark = False

# 0. Useful functions

In this section we define all the functions we need, from creating and training the models to generating the datasets or cleaning unused memory.

### 0.1. Memory management

In [3]:
# Here we define a function to free all possible memory from the GPU to avoid memory issues when dealing with a lot of models
def free_gpu_cache(show_usage = False):
# This function is used to clear the GPU cache and avoid memory problems when dealing with large populations and big models
    if show_usage:
        print("Initial GPU Usage")
        gpu_usage()
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()
    if show_usage:
        print("GPU Usage after emptying the cache")
        gpu_usage()

### 0.2. Model training functions

In [4]:
def trainloop(base_model, transforms, max_epochs = 256, lr = 1e-3, patience = 0, model_save_path = "", verbose = False):
    try:
        accs = {"train" : [],
                "val" : [],
                "test" : []
               }
        times = []
        with tqdm(total = len(transforms), leave = False, desc='Data Augmentation scheme') as pbar0:
            for t_idx, t in enumerate(transforms):
                model = copy.deepcopy(base_model)
                
                if t_idx == len(transforms)-1: # For the no Data Aug. with reinitialized weights case
                    model.apply(weight_reset)
                
                start = time.time()
                train_loader, val_loader = retrieve_datasets(t)
        
                model.to(model.device)
                optimizer = optim.AdamW(model.parameters(), lr)
                
                model.train()
                training_loss = []
                validation_loss = []
                overfit = 0
                best_loss = np.inf
                with tqdm(total = max_epochs , leave = False, desc='Training model') as pbar1:
                    for epoch in range(max_epochs):
                        running_loss = 0.0
                        for inputs, labels in train_loader:
                            inputs = t(inputs)
                            time_start = time.time()
                            inputs, labels = inputs.to(model.device), labels.to(model.device)
                            optimizer.zero_grad()
                            outputs = model(inputs)
                            
                            loss = criterion(outputs, labels)
                            loss.backward()
                            optimizer.step()
                            running_loss += loss.item()
        
                            if time.time() - time_start > model.max_iter_time*4:
                                try: pbar0.update(np.inf)
                                except: pass
                                try: pbar1.update(np.inf)
                                except: pass
                                del model, optimizer
                                free_gpu_cache()
                                raise Exception("This model would take too much time to train. Trying another random model.")
                        training_loss.append(running_loss/len(train_loader))
                        
                        model.eval()
                        running_loss = 0.
                        with tqdm(total=len(val_loader), leave = False, desc='Calculating validation loss') as pbar3:
                            with torch.no_grad():
                                for inputs, labels in val_loader:
                                    inputs, labels = inputs.to(model.device), labels.to(model.device)
                                    outputs = model.forward(inputs)
                                    loss = criterion(outputs, labels)
                                    running_loss += loss.detach().cpu().numpy()
                                    pbar3.update(1)
                            validation_loss.append(running_loss/len(val_loader))
                
                        if patience > 0:
                            if validation_loss[-1] < best_loss:
                                best_loss = validation_loss[-1]
                                overfit = 0
                                torch.save(model.state_dict(), "training_model.pt")
                            else:
                                overfit += 1
                            if overfit >= patience:
                                pbar1.update(max_epochs)
                                break
                        pbar1.update(1)
    
                if verbose:
                    plt.plot(training_loss)
                    plt.plot(validation_loss)
                    plt.legend(["Train","Val"])
                    plt.show()
                if patience > 0:
                    model.load_state_dict(torch.load("training_model.pt"))
    
                running_loss = 0.
                with tqdm(total=len(test_loader), leave = False, desc='Calculating test loss') as pbar3:
                    with torch.no_grad():
                        for inputs, labels in test_loader:
                            inputs, labels = inputs.to(model.device), labels.to(model.device)
                            outputs = model.forward(inputs)
                            loss = criterion(outputs, labels)
                            running_loss += loss.detach().cpu().numpy()
                            pbar3.update(1)
        
                accs["train"] = accs["train"] + copy.deepcopy([training_loss[-1]])
                accs["val"] = accs["val"] + copy.deepcopy([validation_loss[-1]])
                accs["test"] = accs["test"] + copy.deepcopy([running_loss/len(test_loader)])
                
                torch.save(model, "data_aug_"+str(t_idx)+".pt")
                
                del model, optimizer, loss, inputs, labels, outputs
                times += [time.time()-start]
                
                pbar0.update()
                free_gpu_cache()
        
        #valid = np.sum([accs["val"][i]>0.2 for i in range(len(transforms))]) == len(transforms)
        for t_idx in range(len(transforms)):
            shutil.copyfile("data_aug_"+str(t_idx)+".pt", model_save_path+"data_aug_"+str(t_idx)+"/"+str(len(os.listdir(model_save_path+"data_aug_"+str(t_idx))))+".pt")
        return accs, times
    
    except Exception as e:
        try: pbar0.update(np.inf)
        except: pass
        try: pbar1.update(np.inf)
        except: pass
        print("Error during training", e)
        del model, optimizer
        free_gpu_cache()
        raise Exception()


### 0.3. Model generation

In [5]:
class conv_block(nn.Module):
    def __init__(self, in_c, out_c, kernel_size = 3):
        self.kernel_size = kernel_size
        padding = int((kernel_size + 1)/2)
        self.padding = padding
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU()
        
    def forward(self, inputs):
        x = self.conv(inputs)
        x = self.bn(x)
        x = self.relu(x)
        return x  

class block(nn.Module):
    def __init__(self, in_c, out_c, resizing_type = None, from_block_channels = 0, kernel_size = 3):
        super().__init__()
        self.resizing_type = resizing_type
        self.kernel_size = kernel_size
        self.in_c = in_c
        self.out_c = out_c
        self.from_block_channels = from_block_channels
        
        if resizing_type == "D":
            self.resizing = nn.MaxPool2d((2, 2))
        elif resizing_type == "U":
            self.resizing = nn.ConvTranspose2d(in_c, in_c, kernel_size=2, stride=2, padding=0)
        else:
            self.resizing = nn.Identity()
        
        self.conv = conv_block(from_block_channels+in_c, out_c, kernel_size = kernel_size)
        
    def forward(self, inputs, skip):
        x = self.resizing(inputs)
        
        for extra_input in skip:
            x = torch.cat([x, nn.functional.interpolate(extra_input, size = [x.shape[2], x.shape[3]])], axis=1)
        x = self.conv(x)
        return x

class optimized_network(nn.Module):
    """
    A Convolutional Neural Network generated randomly.
    """
    def __init__(self, dataloader, n_blocks = 1, task = "infere", out_size = None, logmin_c = 5, logmax_c = 8, max_k = 9, complex_classifier = False, max_iter_time = 1, verbose = False):
        super().__init__()
        
        if hasattr(n_blocks, '__iter__'):
            if len(n_blocks) not in [1,2]:
                raise Exception("Invalid range for model initial layers.")
        elif n_blocks == int(n_blocks):
            n_blocks = [int(n_blocks), int(n_blocks)]
        else:
            raise Exception("Model initial layers needs to be either a scalar or a scalar range.")

        self.dataloader = dataloader
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.logmin_c = logmin_c
        self.logmax_c = logmax_c
        self.max_k = max_k
        self.complex_classifier = complex_classifier
        self.max_iter_time = max_iter_time
        
        batch = next(iter(dataloader))
        self.in_size = batch[0].shape
        self.label_size = batch[1].shape
        
        if task not in ["image_to_image", "image_to_mask", "object_detection", "regression", "classification", "infere"]:
            print("Unknown task. Infering task from dataset.")
            task = "infere"
        if task == "infere":
            if batch[0].shape[-2:] == batch[1].shape[-2:]:
                if batch[1].is_floating_point():
                    self.task = "image_to_image"
                else:
                    if batch[1].shape[-3] == 1:
                        self.task = "image_to_mask"
                    else:
                        self.task = "object_detection"
            else:
                if batch[1].is_floating_point():
                    self.task = "regression"
                else:
                    self.task = "classification"
        else:
            self.task = task
        
        if out_size:
            self.out_size = out_size
        else:
            if self.task == "classification" or self.task == "object_detection":
                targets = dataloader.dataset.targets
                self.out_size = max(targets) + 1
            else:
                self.out_size = batch[1].shape[1:]
                if len(batch[1].shape)==1: self.out_size = [1]
                
        
        invalid = self.initialize_network(n_blocks = random.randint(n_blocks[0], n_blocks[1]), verbose = verbose)
        if invalid:
            del self.features, self.fc
            free_gpu_cache()

    def calculate_possible_connections(self):
        updown = [0]
        for l in self.features:
            if type(l.resizing) == nn.Identity:
                r = 0
            elif type(l.resizing) == nn.ConvTranspose2d:
                r = 1
            elif type(l.resizing) == nn.MaxPool2d:
                r = -1
            updown = updown + [updown[-1] + r]
        updown = updown[1:]
    
        possible_connections = []
        for idxin, lin in enumerate(updown):
            for idxout, lout in enumerate(updown[idxin+2:]):
                if lin == lout:
                    possible_connections += [(idxin, idxin+2+idxout)]
        
        return possible_connections
    
    def correct_blocks(self):
        for l, blk in enumerate(list(self.features.children())): # We check each block's new input size, rebuilding it if necessary
            if l == 0:
                if not blk.in_c == self.in_size[1]:
                    self.features[l] = block(in_c = self.in_size[1], out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = 0, kernel_size = blk.kernel_size)
    
            else:
                from_blocks = [i for i, x in enumerate(self.connections) if l in x]
                from_block_channels = 0
                for from_block in from_blocks:
                    from_block_channels += self.features[from_block].out_c
                
                if not (blk.in_c == self.features[l-1].out_c and blk.from_block_channels == from_block_channels):
                    self.features[l] = block(in_c = self.features[l-1].out_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size)
        
        self.build_fc()
    
    def initialize_network(self, n_blocks, verbose):
        try:
            self.connections = [[]] * n_blocks
            self.features = nn.Sequential()
            self.genome = []
            for l in range(n_blocks):
                if l == 0:
                    in_c = self.in_size[1]
                else:
                    in_c = out_c

                if l == n_blocks-1 and self.task not in ["classification", "regression"]:
                    out_c = self.out_size[0]
                else:
                    out_c = 2**random.randint(self.logmin_c, self.logmax_c)
                
                resizing_type = random.choice(["U", "D", ""])
                
                kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
                self.features = nn.Sequential(*list(self.features.children()), block(in_c = in_c, out_c = out_c, resizing_type = resizing_type, from_block_channels = 0, kernel_size = kernel_size))

            possible_connections = self.calculate_possible_connections()
            k = random.randint(0, len(possible_connections))
            initial_connections = random.sample(possible_connections, k=k)

            for c in initial_connections:
                self.connections[c[0]] = self.connections[c[0]] + [c[1]]
            
            for i in range(len(self.connections)):
                self.connections[i].sort() # We order the connections so that they are tidy

            self.correct_blocks()
            return False
        except Exception as e:
            return True

    def build_fc(self):
        self.fc = nn.Sequential(nn.Identity())
        if self.task in ["classification", "regression"]:
            feat_size = np.prod([*self.to(self.device).forward(torch.zeros(self.in_size, device = self.device)).shape[1:]])
            free_gpu_cache()
            if self.task == "classification":
                if self.complex_classifier:
                    self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size*4), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*4, self.out_size*2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*2, self.out_size[0]), nn.LogSoftmax(dim=0))
                else:
                    self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size[0]), nn.LogSoftmax(dim=0))
            else:
                if self.complex_classifier:
                    self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size*4), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*4, self.out_size*2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*2, self.out_size[0]))
                else:
                    self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size[0]))

    def forward(self, inputs):
        skip = [[]]*len(self.features)
        x = inputs
        for i, l in enumerate(self.features):
            x = l(x, skip[i])
            for to_block in self.connections[i]:
                skip[to_block] = skip[to_block]+[x]
        if (self.task == "classification" or self.task == "regression"):
            x = torch.flatten(x, start_dim=1)
            for l in self.fc:
                x = l(x)
        else:
            x = nn.functional.interpolate(x, size = [inputs.shape[2], inputs.shape[3]])
        del skip
        return x

    def mutate(self, mut_amount = 1, probs = {"add_b" : 0.4, "delete_b" : 0.4, "modify_b" : 0.2}, verbose = False):
        operations = list(probs.keys())
        
        backup_features = copy.deepcopy(self.features)
        backup_fc = copy.deepcopy(self.fc)
        backup_connections = copy.deepcopy(self.connections)
        
        valid_model = False
        while not valid_model:
            try:
                mutations = random.choices(operations, weights = [probs[operation] for operation in operations], k = mut_amount)
                for mut in mutations:
                    match mut:
                        case "add_b":
                            self.add_block(verbose)
                        case "delete_b":
                            if len(self.features) > 1: # If we can't delete any block, we add one instead
                                self.delete_block(verbose)
                            else:
                                self.add_block(verbose)
                        case "modify_b":
                            self.modify_block(verbose)
    
                for i in range(len(self.connections)):
                    self.connections[i].sort() # We order the connections so that they are tidy
                
                self.build_fc()
                valid_model = True
                
            except Exception as e:
                if verbose:
                    print(e)
                self.features = copy.deepcopy(backup_features)
                self.fc = copy.deepcopy(backup_fc)
                self.connections = copy.deepcopy(backup_connections)
    
    def add_block(self, verbose = False):
        # We chose a random block to add another one after
        index = random.randint(0, len(self.features)-1)
        if verbose:
            print("Added block between blocks", index, "and " + str(index+1)+".")

        self.connections = self.connections[:index+1] + [[]] + self.connections[index+1:] # We add space for the new block to make connections
        self.connections = [[b[c]*(b[c] <= index) + (b[c]+1)*((b[c]) > index) for c in range(len(b))] for b in self.connections] # And we adjust the indexes of all connections

        # We chose the hyperparameters for the new block
        if self.task not in ["classification", "regression"]:
            resizing_type = "" # If we need the output size to remain constant we do not add a resizing
        else:
            resizing_type = random.choice(["U", "D", ""])
        kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
        
        features = nn.Sequential(*list(self.features.children())[:index+1], # We keep the previous blocks, add a block with in and out channels equal to the output channels of the previous block
                                  block(in_c = self.features[index].out_c, out_c = self.features[index].out_c, resizing_type = resizing_type, from_block_channels = 0, kernel_size = kernel_size),
                                 *list(self.features.children())[index+1:]) # And then add the remaining blocks
        
        self.features = features

    def delete_block(self, verbose = False):
        # We chose a random block to delete
        index = random.randint(0, len(self.features)-1)
        if verbose:
            print("Block", index, "deleted.")

        self.connections.pop(index) # We delete the skip connections of the deleted block
        self.connections = [[b[c]*(b[c] < index) + (b[c]-1)*((b[c]) > index) for c in range(len(b)) if index != b[c]] for b in self.connections] # We adjust the indexes of all connections
        self.connections = [[b[c] for c in range(len(b)) if b[c]-1 != i] for i, b in enumerate(self.connections)] # And we remove the skip connections btween contiguous blocks

        if index == len(self.features)-1: # If it's the last block, we just remove it
            self.features = nn.Sequential(*list(self.features.children())[:index])
        else: # If not, we have to modify the input channels of the subsequent block and add the rest of the blocks onwards, taking the new skipping connections into consideration
            from_blocks = [i for i, x in enumerate(self.connections) if index in x]
            from_block_channels = 0
            for from_block in from_blocks:
                from_block_channels += self.features[from_block].out_c
                    
            features = nn.Sequential(*list(self.features.children())[:index],
                                          block(in_c = self.features[index].in_c, out_c = self.features[index+1].out_c, resizing_type = self.features[index+1].resizing_type, from_block_channels = from_block_channels, kernel_size = self.features[index+1].kernel_size))
            
            for l, blk in enumerate(list(self.features.children())[index+2:]):
                from_blocks = [i for i, x in enumerate(self.connections) if index+1+l in x]
                from_block_channels = 0
                for from_block in from_blocks:
                    from_block_channels += features[from_block].out_c

                features = nn.Sequential(*list(features.children()), block(in_c = blk.in_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size))

            self.features = features

    def modify_block(self, verbose = False):
        # We chose a random block to modify
        index = random.randint(0, len(self.features)-1)
        if verbose:
            print("Block", index, "modified.")

        out_c = 2**random.randint(self.logmin_c, self.logmax_c)
        resizing_type = random.choice(["U", "D", ""])
        kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
        features = nn.Sequential(*list(self.features.children())[:index], # We keep the previous blocks and add a modified block
                                  block(in_c = self.features[index].in_c, out_c = out_c, resizing_type = resizing_type, from_block_channels = self.features[index].from_block_channels, kernel_size = kernel_size))
        
        for l, blk in enumerate(list(self.features.children())[index+1:]): # We add the next blocks, modifying the connections accordingly and adapting the input channels on the first one
            from_blocks = [i for i, x in enumerate(self.connections) if index+1+l in x]
            from_block_channels = 0
            for from_block in from_blocks:
                from_block_channels += features[from_block].out_c

            features = nn.Sequential(*list(features.children()),
                                     block(in_c = out_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size))

            out_c = blk.out_c
        
        self.features = features

def weight_reset(layer):
    if hasattr(layer, 'reset_parameters'):
        layer.reset_parameters()

### 0.5. Dataset retrieval with Data Augmentation

In [6]:
#od.download("https://www.kaggle.com/datasets/sumission9automl2024/ct-misalignments-dataset")

In [7]:
class VolDset(Dataset):
    def __init__(self, path, view, transform):
        #Copy attributes of the function into self variable
        self.path_inp = path
        self.inp = os.listdir(self.path_inp)
        self.n_slices=0
        self.view=view
        self.transform=transform
        self.path=path

    def __getitem__(self, idx):
        pat=r'\d+x\d+x\d+'
        self.vol_inp = self.inp[idx]
        self.size = re.findall(pat,str(self.vol_inp))
        self.size = self.size[0].split("x")
        self.size = [int(i) for i in self.size]
        
        self.vol_inp = open(str(self.path_inp)+'/'+self.vol_inp,'rb') #only opens the file for reading
        self.vol_inp = np.fromfile(self.vol_inp,dtype='<f4')
        self.vol_inp = self.vol_inp.reshape(self.size[0], self.size[1], self.size[2], order='F')
        
        self.param_out = self.inp[idx]
        pat=r'-*\d+\.\d+mm_du+' # pat=r'-*\d+\.\d+deg_roll+' 
        self.param_out = re.findall(pat,str(self.param_out))
        self.param_out = float(self.param_out[0][:-5]) #8
        
        self.rat_number = self.inp
        pat=r'[RAaTt_]{4,5}\d\w'
        self.rat_number = re.findall(pat,str(self.rat_number))
        self.rats = list(set(self.rat_number))
        self.rat_number = self.rats.index(self.rat_number[idx])
        
        if self.transform:
            if self.view=='axial':
                self.list_img_inp = [self.transform(self.vol_inp[:,:,i]) for i in range(self.size[2])]
            elif self.view=='coronal':
                self.list_img_inp = [self.transform(self.vol_inp[:,i,:]) for i in range(self.size[1])]
        
        return self.list_img_inp, self.param_out, self.rat_number
    
    def __len__(self):            
        return len(self.inp)

In [8]:
path_data = Path(os.getcwd()+r"/ct_misalignments_dataset/dataset")

view='axial'
    
data_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((256,256)),
        transforms.Lambda(lambda x: torch.cat([x, x, x], 0)),
        transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])])

image_datasets_unsqueeze = VolDset(path_data,view,data_transforms)
dataset_sizes = len(image_datasets_unsqueeze)
print("The dataset contains", dataset_sizes, "volumes")

The dataset contains 150 volumes


In [9]:
if not os.path.exists("ct_misalignments_dataset\\split_dataset"):
    os.makedirs("ct_misalignments_dataset\\split_dataset\\train")
    os.makedirs("ct_misalignments_dataset\\split_dataset\\val")
    os.makedirs("ct_misalignments_dataset\\split_dataset\\test")
    x_tr = []
    x_val = []
    x_test = []
    y_tr = []
    y_val = []
    y_test = []
    
    # We can divide our dataset by this amount if we lack memory
    dataset_red = 1;
    
    # We set 5/7 rats to be our training dataset, 1/7 to be our validation dataset and 1/7 to be our test dataset
    train_or_val = ['test']*1 + ['train']*4 + ['val']*1 # ['test']*1 + ['val']*1 + ['train']*4
    # We split our volumes into slices and pack them together
    with tqdm(total=int(dataset_sizes/dataset_red), leave = False, desc='Converting volumes') as pbar:
        for i in range(int(dataset_sizes/dataset_red)):
            inp,lbl,rat=image_datasets_unsqueeze[dataset_red*i]
            pbar.update(1)
            for j in range(len(inp)):
                if train_or_val[rat]=='train':
                    torch.save(inp[j], "ct_misalignments_dataset\\split_dataset\\train\\rat"+str(rat)+"_slice"+str(j)+"_"+str(lbl)+".pt")
                elif train_or_val[rat]=='val':
                    torch.save(inp[j], "ct_misalignments_dataset\\split_dataset\\val\\rat"+str(rat)+"_slice"+str(j)+"_"+str(lbl)+".pt")
                else:
                    torch.save(inp[j], "ct_misalignments_dataset\\split_dataset\\test\\rat"+str(rat)+"_slice"+str(j)+"_"+str(lbl)+".pt")
        print("Dataset succesfully converted")
    
    del inp, lbl, rat, image_datasets_unsqueeze
else: print("Dataset already converted")

Dataset already converted


In [10]:
class SliceDset(Dataset):
    def __init__(self, path, transforms = None, repeats = 1):
        #Copy attributes of the function into self variable
        self.path = path
        self.inp = os.listdir(self.path)*repeats
        self.out = [float(p.split("_")[-1].split(".pt")[0]) for p in self.inp]
        self.transforms=transforms

    def __getitem__(self, idx):
        self.slice = torch.load(self.path+"\\"+self.inp[idx])
        self.label = torch.tensor(self.out[idx])
        
        if self.transforms: self.slice = self.transforms(self.slice)
        
        return self.slice, self.label
    
    def __len__(self):            
        return len(self.inp)
        
def retrieve_datasets(transforms, val_da = 16):
    train_dset = SliceDset("ct_misalignments_dataset\\split_dataset\\train", transforms)
    train_loader = DataLoader(train_dset, batch_size=16, shuffle=True)
    valid_dset = SliceDset("ct_misalignments_dataset\\split_dataset\\val", transforms, val_da)
    val_loader = DataLoader(valid_dset, batch_size=16, shuffle=False)
    return train_loader, val_loader

# 1. Mutated model generation for Automatic Callibration

In this section, models will be generated at by mutating the best performing models from the previous test and trained with 3 Data Augmentation schemes besides weight reinitialization to compare with a non-augmented training.

### 1.1. Define the transforms with and without data augmentation

In [11]:
transforms_vanilla = v2.Compose([v2.ToTensor(),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug1 = v2.Compose([v2.ToTensor(),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                 v2.Lambda(lambda x : x + (500*random.random())*torch.randn(x.shape)),
                                ])

transforms_dataug2 = v2.Compose([v2.ToTensor(),
                                 v2.RandomVerticalFlip(p=1),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug3 = v2.Compose([v2.ToTensor(),
                                 v2.GaussianBlur(kernel_size=9, sigma=(0.000001,1)),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms = [transforms_vanilla, transforms_dataug1, transforms_dataug2, transforms_dataug3, transforms_vanilla] # The second transforms_vanilla is for the reinitialized weights case

# Load the test data, with no augmentations
test_dset=SliceDset("ct_misalignments_dataset\\split_dataset\\test")
test_loader = DataLoader(test_dset, batch_size=16, shuffle=True)

criterion = nn.MSELoss()

### 1.2. Create similar models via single mutation operation

In [12]:
# Model and training parameters
n_tests = 16
max_layers = 12
max_epochs = 256
patience = 3
lr = 1e-4
seed_value = 0

model_args = {"dataloader" : test_loader, # To check the input and output sizes, type of problem, etc, from
              "task" : "regression",
              "out_size" : None,
              "logmin_c" : 4, # Number of channels per layer goes from 16 to 1024
              "logmax_c" : 10,
              "max_k" : 7, # Kernel size is at most 7 px wide
              "max_iter_time" : 2, # Only train models that require at most 2 seconds per batch
              "verbose" : True}

In [13]:
#model_save_path = "saved_models/callibration_mutation_original/"
#orig_models = "saved_models/callibration/"

## Load results from previous test
#if os.path.exists('tr_accs.json'):
#    with open('tr_accs.json', 'r') as file:
#        tr_accs = json.load(file)
#    with open('val_accs.json', 'r') as file:
#        val_accs = json.load(file)
#    with open('test_accs.json', 'r') as file:
#        test_accs = json.load(file)
#    with open('train_times.json', 'r') as file:
#        train_times = json.load(file)
#        
#test_van = np.array([test for test in test_accs["vanilla"] if test != None])
#test_da1 = np.array([test for test in test_accs["daug1"] if test != None])
#test_da2 = np.array([test for test in test_accs["daug2"] if test != None])
#test_da3 = np.array([test for test in test_accs["daug3"] if test != None])
#test_rew = np.array([test for test in test_accs["reinit"] if test != None])
#
#best_idx = np.argmin((test_van+test_da1+test_da2+test_da3+test_rew)/5)
#best_model = os.listdir(orig_models+"data_aug_0")[best_idx]
#
#for t_idx in range(len(transforms)):
#    os.makedirs(model_save_path+"data_aug_"+str(t_idx), exist_ok=True)
#    shutil.copyfile(orig_models+"data_aug_"+str(t_idx)+"/"+best_model, model_save_path+"data_aug_"+str(t_idx)+"/0.pt")
#
#best_idx = np.argwhere(test_accs["vanilla"]==test_van[best_idx])[0][0]
#tr_accs = {"vanilla" : [tr_accs["vanilla"][best_idx]],"daug1" : [tr_accs["daug1"][best_idx]],"daug2" : [tr_accs["daug2"][best_idx]],"daug3" : [tr_accs["daug3"][best_idx]], "reinit" : [tr_accs["reinit"][best_idx]]}
#val_accs = {"vanilla" : [val_accs["vanilla"][best_idx]],"daug1" : [val_accs["daug1"][best_idx]],"daug2" : [val_accs["daug2"][best_idx]],"daug3" : [val_accs["daug3"][best_idx]], "reinit" : [val_accs["reinit"][best_idx]]}
#test_accs = {"vanilla" : [test_accs["vanilla"][best_idx]],"daug1" : [test_accs["daug1"][best_idx]],"daug2" : [test_accs["daug2"][best_idx]],"daug3" : [test_accs["daug3"][best_idx]], "reinit" : [test_accs["reinit"][best_idx]]}
#train_times = {"vanilla" : [train_times["vanilla"][best_idx]],"daug1" : [train_times["daug1"][best_idx]],"daug2" : [train_times["daug2"][best_idx]],"daug3" : [train_times["daug3"][best_idx]], "reinit" : [train_times["reinit"][best_idx]]}
#Path("tr_accs_original.json").write_text(json.dumps(tr_accs))
#Path("val_accs_original.json").write_text(json.dumps(val_accs))
#Path("test_accs_original.json").write_text(json.dumps(test_accs))
#Path("train_times_original.json").write_text(json.dumps(train_times));

In [14]:
torch.cuda.set_per_process_memory_fraction(0.85, device=0)

In [15]:
base_model_path = "saved_models/callibration_mutation_original/"
model_save_path = "saved_models/callibration_1mutation/"

for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)

if os.path.exists('tr_accs_1mut.json'):
    with open('tr_accs_1mut.json', 'r') as file:
        tr_accs = json.load(file)
    with open('val_accs_1mut.json', 'r') as file:
        val_accs = json.load(file)
    with open('test_accs_1mut.json', 'r') as file:
        test_accs = json.load(file)
    with open('train_times_1mut.json', 'r') as file:
        train_times = json.load(file)
else:
    tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

seed_value = len(tr_accs["vanilla"])**2
with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
    pbar_tests.update(len(val_accs["vanilla"]))
    while len(val_accs["vanilla"]) < n_tests:
        seed_value += 1
        try:
            free_gpu_cache()
            # Now we copy the original model and mutate it just once before training
            set_seed(seed_value)
            base_model = torch.load(base_model_path+"data_aug_0/0.pt", weights_only=False)
            base_model.mutate(mut_amount = 1, probs = {"add_b" : 0.4, "delete_b" : 0.4, "modify_b" : 0.3}, verbose = True)
            
            accs, times = trainloop(base_model, transforms, max_epochs = max_epochs, lr = lr, patience = patience, model_save_path = model_save_path)
            if not accs == None:
                print("Attained accuracies:", accs)
                print("In seconds:", times)
                tr_accs["vanilla"] = tr_accs["vanilla"] + [float(accs["train"][0])]
                tr_accs["daug1"] = tr_accs["daug1"] + [float(accs["train"][1])]
                tr_accs["daug2"] = tr_accs["daug2"] + [float(accs["train"][2])]
                tr_accs["daug3"] = tr_accs["daug3"] + [float(accs["train"][3])]
                tr_accs["reinit"] = tr_accs["reinit"] + [float(accs["train"][4])]
                
                val_accs["vanilla"] = val_accs["vanilla"] + [float(accs["val"][0])]
                val_accs["daug1"] = val_accs["daug1"] + [float(accs["val"][1])]
                val_accs["daug2"] = val_accs["daug2"] + [float(accs["val"][2])]
                val_accs["daug3"] = val_accs["daug3"] + [float(accs["val"][3])]
                val_accs["reinit"] = val_accs["reinit"] + [float(accs["val"][4])]
                
                test_accs["vanilla"] = test_accs["vanilla"] + [float(accs["test"][0])]
                test_accs["daug1"] = test_accs["daug1"] + [float(accs["test"][1])]
                test_accs["daug2"] = test_accs["daug2"] + [float(accs["test"][2])]
                test_accs["daug3"] = test_accs["daug3"] + [float(accs["test"][3])]
                test_accs["reinit"] = test_accs["reinit"] + [float(accs["test"][4])]
    
                train_times["vanilla"] = train_times["vanilla"] + [float(times[0])]
                train_times["daug1"] = train_times["daug1"] + [float(times[1])]
                train_times["daug2"] = train_times["daug2"] + [float(times[2])]
                train_times["daug3"] = train_times["daug3"] + [float(times[3])]
                train_times["reinit"] = train_times["reinit"] + [float(times[4])]
    
                Path("tr_accs_1mut.json").write_text(json.dumps(tr_accs))
                Path("val_accs_1mut.json").write_text(json.dumps(val_accs))
                Path("test_accs_1mut.json").write_text(json.dumps(test_accs))
                Path("train_times_1mut.json").write_text(json.dumps(train_times))
    
                pbar_tests.update()
            else:
                print("Generated useless model, retrying")
                
        except Exception as e:
            print(e)

Generating and training models:   0%|          | 0/16 [00:00<?, ?it/s]

Block 0 deleted.


Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Error during training CUDA out of memory. Tried to allocate 4.13 GiB. GPU 0 has a total capacity of 24.00 GiB of which 3.69 GiB is free. 20.40 GiB allowed; Of the allocated memory 14.99 GiB is allocated by PyTorch, and 3.51 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Block 3 deleted.


...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Calculating validation loss:   0%|          | 0/5000 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Calculating validation loss:   0%|          | 0/5000 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [16]:
print("tr_accs_1mut =", tr_accs)
print("val_accs_1mut =", val_accs)
print("test_accs_1mut =", test_accs)
print("train_times_1mut =", train_times)

tr_accs_1mut = {'vanilla': [0.28646734523773193, 0.2867969605743885, 0.2867211029827595, 0.28646734523773193, 0.2867969605743885, 0.2867211029827595, 0.2866113100349903, 0.286665275323391, 0.3217372987568378, 0.2870572353422642, 0.28658614003658295, 0.28634376488924024, 0.32278193016648293, 0.2872452960193157, 0.28677590116262436, 0.28738624564409254], 'daug1': [0.2861122602701187, 0.28699469947814943, 0.2864523933172226, 0.2861122602701187, 0.28699469947814943, 0.2864523933172226, 0.2863212842106819, 0.2865227021276951, 0.2884033505797386, 0.2865116435348988, 0.32923370962142945, 0.2862115763962269, 0.28734833102822305, 0.28737719175815585, 0.28619829947948455, 0.28704158319830897], 'daug2': [0.2863138601064682, 0.28709878624081614, 0.2865015801012516, 0.2863138601064682, 0.28709878624081614, 0.2865015801012516, 0.2864122482776642, 0.2864317581653595, 0.2875380614221096, 0.28750587679743767, 0.3371614780604839, 0.28636584072113036, 0.28878197222352026, 0.2906079964876175, 0.2862122928

In [ ]:
base_model_path = "saved_models/callibration_mutation_original/"
model_save_path = "saved_models/callibration_3mutations/"
os.makedirs(model_save_path, exist_ok = True)
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)


if os.path.exists('tr_accs_3mut.json'):
    with open('tr_accs_3mut.json', 'r') as file:
        tr_accs = json.load(file)
    with open('val_accs_3mut.json', 'r') as file:
        val_accs = json.load(file)
    with open('test_accs_3mut.json', 'r') as file:
        test_accs = json.load(file)
    with open('train_times_3mut.json', 'r') as file:
        train_times = json.load(file)
else:
    tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

seed_value = 4200
with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
    pbar_tests.update(len(val_accs["vanilla"]))
    while len(val_accs["vanilla"]) < n_tests:
        seed_value -= 1
        try:
            free_gpu_cache()
            # Now we copy the original model and mutate it just once before training
            set_seed(seed_value)
            base_model = torch.load(base_model_path+"data_aug_0/0.pt", weights_only=False)
            base_model.mutate(mut_amount = 3, probs = {"add_b" : 0.4, "delete_b" : 0.4, "modify_b" : 0.3}, verbose = True)
            
            accs, times = trainloop(base_model, transforms, max_epochs = max_epochs, lr = lr, patience = patience, model_save_path = model_save_path)
            if not accs == None:
                print("Attained accuracies:", accs)
                print("In seconds:", times)
                tr_accs["vanilla"] = tr_accs["vanilla"] + [float(accs["train"][0])]
                tr_accs["daug1"] = tr_accs["daug1"] + [float(accs["train"][1])]
                tr_accs["daug2"] = tr_accs["daug2"] + [float(accs["train"][2])]
                tr_accs["daug3"] = tr_accs["daug3"] + [float(accs["train"][3])]
                tr_accs["reinit"] = tr_accs["reinit"] + [float(accs["train"][4])]
                
                val_accs["vanilla"] = val_accs["vanilla"] + [float(accs["val"][0])]
                val_accs["daug1"] = val_accs["daug1"] + [float(accs["val"][1])]
                val_accs["daug2"] = val_accs["daug2"] + [float(accs["val"][2])]
                val_accs["daug3"] = val_accs["daug3"] + [float(accs["val"][3])]
                val_accs["reinit"] = val_accs["reinit"] + [float(accs["val"][4])]
                
                test_accs["vanilla"] = test_accs["vanilla"] + [float(accs["test"][0])]
                test_accs["daug1"] = test_accs["daug1"] + [float(accs["test"][1])]
                test_accs["daug2"] = test_accs["daug2"] + [float(accs["test"][2])]
                test_accs["daug3"] = test_accs["daug3"] + [float(accs["test"][3])]
                test_accs["reinit"] = test_accs["reinit"] + [float(accs["test"][4])]
    
                train_times["vanilla"] = train_times["vanilla"] + [float(times[0])]
                train_times["daug1"] = train_times["daug1"] + [float(times[1])]
                train_times["daug2"] = train_times["daug2"] + [float(times[2])]
                train_times["daug3"] = train_times["daug3"] + [float(times[3])]
                train_times["reinit"] = train_times["reinit"] + [float(times[4])]
    
                Path("tr_accs_3mut.json").write_text(json.dumps(tr_accs))
                Path("val_accs_3mut.json").write_text(json.dumps(val_accs))
                Path("test_accs_3mut.json").write_text(json.dumps(test_accs))
                Path("train_times_3mut.json").write_text(json.dumps(train_times))
    
                #pbar_tests.update()
            else:
                print("Generated useless model, retrying")
                
        except Exception as e:
            print(e)


Generating and training models:   0%|          | 0/16 [00:00<?, ?it/s]

Block 3 deleted.
Added block between blocks 2 and 3.
Block 2 deleted.


Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Calculating validation loss:   0%|          | 0/5000 [00:00<?, ?it/s]

...

Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Calculating validation loss:   0%|          | 0/5000 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Calculating validation loss:   0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
print("tr_accs_3mut =", tr_accs)
print("val_accs_3mut =", val_accs)
print("test_accs_3mut =", test_accs)
print("train_times_3mut =", train_times)

In [ ]:
base_model = "saved_models/callibration_mutation_original/"
model_save_path = "saved_models/callibration_1mutation/"
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)

if os.path.exists('tr_accs_1mut.json'):
    with open('tr_accs_1mut.json', 'r') as file:
        tr_accs = json.load(file)
    with open('val_accs_1mut.json', 'r') as file:
        val_accs = json.load(file)
    with open('test_accs_1mut.json', 'r') as file:
        test_accs = json.load(file)
    with open('train_times_1mut.json', 'r') as file:
        train_times = json.load(file)
else:
    tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

seed_value = len(val_accs["vanilla"])
#with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
while len(val_accs["vanilla"]) < n_tests:
    seed_value += 1
    try:
        free_gpu_cache()
        # Now we copy the original model and mutate it just once before training
        accs, times = mut_trainloop(base_model, mutations = 1, seed = seed_value, transforms = transforms, max_epochs = max_epochs, lr = lr, patience = patience, model_save_path = model_save_path)
        if not accs == None:
            tr_accs["vanilla"] = tr_accs["vanilla"] + [accs["train"][0]]
            tr_accs["daug1"] = tr_accs["daug1"] + [accs["train"][1]]
            tr_accs["daug2"] = tr_accs["daug2"] + [accs["train"][2]]
            tr_accs["daug3"] = tr_accs["daug3"] + [accs["train"][3]]
            tr_accs["reinit"] = tr_accs["reinit"] + [accs["train"][4]]
            
            val_accs["vanilla"] = val_accs["vanilla"] + [accs["val"][0]]
            val_accs["daug1"] = val_accs["daug1"] + [accs["val"][1]]
            val_accs["daug2"] = val_accs["daug2"] + [accs["val"][2]]
            val_accs["daug3"] = val_accs["daug3"] + [accs["val"][3]]
            val_accs["reinit"] = val_accs["reinit"] + [accs["val"][4]]
            
            test_accs["vanilla"] = test_accs["vanilla"] + [accs["test"][0]]
            test_accs["daug1"] = test_accs["daug1"] + [accs["test"][1]]
            test_accs["daug2"] = test_accs["daug2"] + [accs["test"][2]]
            test_accs["daug3"] = test_accs["daug3"] + [accs["test"][3]]
            test_accs["reinit"] = test_accs["reinit"] + [accs["test"][4]]

            train_times["vanilla"] = train_times["vanilla"] + [times[0]]
            train_times["daug1"] = train_times["daug1"] + [times[1]]
            train_times["daug2"] = train_times["daug2"] + [times[2]]
            train_times["daug3"] = train_times["daug3"] + [times[3]]
            train_times["reinit"] = train_times["reinit"] + [times[4]]

            Path("tr_accs_1mut.json").write_text(json.dumps(tr_accs))
            Path("val_accs_1mut.json").write_text(json.dumps(val_accs))
            Path("test_accs_1mut.json").write_text(json.dumps(test_accs))
            Path("train_times_1mut.json").write_text(json.dumps(train_times))

            print("Completed training", len(val_accs["vanilla"]), "out of", n_tests)
            #pbar_tests.update()
        else:
            print("Generated useless model, retrying")
            
    except Exception as e:
        print(e)

Added block between blocks 0 and 1.
Succesfully trained model with transform 0 / 5
Added block between blocks 0 and 1.


### 1.3. Resulting accuracies and training times

In [ ]:
print("tr_accs =", tr_accs)
print("val_accs =", val_accs)
print("test_accs =", test_accs)
print("train_times =", train_times)

### 1.4 Create similar models via three mutation operations

In [ ]:
base_model = "saved_models/callibration_mutation_original/"
model_save_path = "saved_models/callibration_3mutations/"
os.makedirs(model_save_path, exist_ok = True)
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)

if os.path.exists('tr_accs_3mut.json'):
    with open('tr_accs_3mut.json', 'r') as file:
        tr_accs = json.load(file)
    with open('val_accs_3mut.json', 'r') as file:
        val_accs = json.load(file)
    with open('test_accs_3mut.json', 'r') as file:
        test_accs = json.load(file)
    with open('train_times_3mut.json', 'r') as file:
        train_times = json.load(file)
else:
    tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

#with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
#pbar_tests.update(len(val_accs["vanilla"]))
while len(val_accs["vanilla"]) < n_tests:
    seed_value += 1
    try:
        free_gpu_cache()
        # Now we copy the original model and mutate 3 times before training
        accs, times = mut_trainloop(base_model, mutations = 3, seed = seed_value, transforms = transforms, max_epochs = max_epochs, lr = lr, patience = patience, model_save_path = model_save_path)
        
        if not accs == None:
            tr_accs["vanilla"] = tr_accs["vanilla"] + [accs["train"][0]]
            tr_accs["daug1"] = tr_accs["daug1"] + [accs["train"][1]]
            tr_accs["daug2"] = tr_accs["daug2"] + [accs["train"][2]]
            tr_accs["daug3"] = tr_accs["daug3"] + [accs["train"][3]]
            tr_accs["reinit"] = tr_accs["reinit"] + [accs["train"][4]]
            
            val_accs["vanilla"] = val_accs["vanilla"] + [accs["val"][0]]
            val_accs["daug1"] = val_accs["daug1"] + [accs["val"][1]]
            val_accs["daug2"] = val_accs["daug2"] + [accs["val"][2]]
            val_accs["daug3"] = val_accs["daug3"] + [accs["val"][3]]
            val_accs["reinit"] = val_accs["reinit"] + [accs["val"][4]]
            
            test_accs["vanilla"] = test_accs["vanilla"] + [accs["test"][0]]
            test_accs["daug1"] = test_accs["daug1"] + [accs["test"][1]]
            test_accs["daug2"] = test_accs["daug2"] + [accs["test"][2]]
            test_accs["daug3"] = test_accs["daug3"] + [accs["test"][3]]
            test_accs["reinit"] = test_accs["reinit"] + [accs["test"][4]]

            train_times["vanilla"] = train_times["vanilla"] + [times[0]]
            train_times["daug1"] = train_times["daug1"] + [times[1]]
            train_times["daug2"] = train_times["daug2"] + [times[2]]
            train_times["daug3"] = train_times["daug3"] + [times[3]]
            train_times["reinit"] = train_times["reinit"] + [times[4]]

            Path("tr_accs_3mut.json").write_text(json.dumps(tr_accs))
            Path("val_accs_3mut.json").write_text(json.dumps(val_accs))
            Path("test_accs_3mut.json").write_text(json.dumps(test_accs))
            Path("train_times_3mut.json").write_text(json.dumps(train_times))

            print("Completed training", len(val_accs["vanilla"]), "out of", n_tests)
            #pbar_tests.update()
        else:
            print("Generated useless model, retrying")
            
    except Exception as e:
        print(e)


### 1.5. Resulting accuracies and training times

In [ ]:
print("tr_accs =", tr_accs)
print("val_accs =", val_accs)
print("test_accs =", test_accs)
print("train_times =", train_times)